# Study 919 — Methodology Shock — the teardown

A market-model event study over a hardcoded calendar of index **rule changes**, priced against a randomisation (placebo) null because the treated sample has seven observations. Inside: the two legs, the nine-window sweep with Bonferroni, the drop-one jackknife, the era cut, the beta-hedged costed pair with a borrow sweep, the deployed-capital race, and the live synthetic control.

Design, in one line: fit `r_treated = a + b * r_control` on the 250 sessions ending 21 sessions before the event, cumulate `r_treated - a - b * r_control` over the window, pool across events. One execution lag — the announcement is public at the day-0 close, the tradable window starts at **+1**.

> 💡 *In plain words:* we ask how much the affected fund moved beyond what its sibling explains, and then check whether random dates do the same thing just as often.

Every real number is frozen from `docs/results.md` (Fingerprint `547814cd71c7`).

In [1]:
R = {'start': '1993-01-29', 'end': '2026-06-30', 'n_days': 8411, 'fp': '547814cd71c7', 'n_events_listed': 8, 'n_events': 7, 'ann_car': -14.1, 'ann_med': -22.8, 'ann_sd': 122.9, 'ann_t': -0.3, 'ann_hac': -0.3, 'ann_hit': 43, 'ann_p': 0.802, 'ann_null_mu': -3.4, 'ann_null_sd': 64.2, 'ann_null_lo': -129, 'ann_null_hi': 124, 'ann_mde': 114, 'ann_boot_lo': -93, 'ann_boot_hi': 73, 'ann_blk_lo': -108, 'ann_blk_hi': 79, 'eff_car': 35.7, 'eff_med': 36.9, 'eff_sd': 81.6, 'eff_t': 1.16, 'eff_hac': 0.86, 'eff_hit': 57, 'eff_p': 0.541, 'eff_mde': 75, 'eff_boot_lo': -15, 'eff_boot_hi': 96, 'eff_blk_lo': -42, 'eff_blk_hi': 114, 'pre_car': -33.2, 'pre_t': -3.43, 'pre_p': 0.451, 'worst_p': 0.423, 'bonf': 1.0, 'n_windows': 9, 'jk_min': -48.5, 'jk_min_t': -1.32, 'jk_max': 9.7, 'era_e_n': 3, 'era_e_car': -60.0, 'era_e_t': -1.23, 'era_l_n': 4, 'era_l_car': 20.4, 'era_l_t': 0.28, 'trade_gross': -7.7, 'trade_net': -29.3, 'trade_t': -0.62, 'trade_win': 43, 'naive_gross': 46.2, 'naive_net': 24.3, 'naive_t': 0.51, 'eff_trade_gross': 43.2, 'eff_trade_net': 21.8, 'eff_trade_t': 0.7, 'cost0_net': -8.2, 'cost25_net': -115.5, 'cost25_t': -2.3, 'fin_rate': 200, 'fin_charge': 0.52, 'fin0_net': -28.8, 'fin500_net': -30.1, 'dc_events': 5, 'dc_dropped': 2, 'live_days': 50, 'total_days': 4802, 'overlay_total': 65, 'overlay_sharpe': 0.187, 'overlay_t': 0.37, 'spy_sharpe': 0.542, 'fix_old_date': '2011-03-24', 'fix_new_date': '2011-04-05', 'fix_old_car': -120.1, 'fix_new_car': 36.3, 'fix_old_pooled': -36.4, 'syn_planted': 250, 'syn_rec': 314.9, 'syn_rec_t': 3.21, 'syn_rec_p': 0.0, 'syn_rec_hit': 92, 'syn_null': 64.9, 'syn_null_p': 0.27, 'syn_seed_mu': 17.4, 'syn_seed_sd': 53.5}

## 1. The headline — pooled CAR on both legs, window [+1,+10]

Beta-adjusted CARs in bps, seven usable events per leg. The placebo *p* comes from 2,000 draws that replace each real date with a random session on the same pair (excluding ±30 sessions around any real event) and recompute identically.

Two caveats stated before the numbers, not after. (a) **The legs are not independent**: both multiple-share-class rows took effect on the day they were announced, so 2 of the 7 observations are shared. They are shown side by side as a sign check and never pooled. (b) **One announcement date in the calendar was wrong** — the 2011 Nasdaq-100 rebalance was dated 2011-03-24 and stamped `exact`; the press release is 2011-04-05. Fixing it moved that event from −120.1 to +36.3 bps and the pooled announcement leg from −36.4 to −14.1 bps. A seven-observation study is exactly that sensitive to its own data entry.

> 💡 *In plain words:* two legs, opposite signs, and both look like random dates.

In [2]:
print(f"announce  : n={R['n_events']}  CAR {R['ann_car']:+.1f} bps  median {R['ann_med']:+.1f}  "
      f"sd {R['ann_sd']:.1f}  hit {R['ann_hit']}%")
print(f"            cross-event t {R['ann_t']:+.2f}  HAC t (daily AR) {R['ann_hac']:+.2f}  "
      f"placebo p {R['ann_p']:.3f}")
print(f"            placebo null {R['ann_null_mu']:+.1f} +/- {R['ann_null_sd']:.1f} bps, "
      f"95% band [{R['ann_null_lo']:+d}, {R['ann_null_hi']:+d}]")
print(f"effective : n={R['n_events']}  CAR {R['eff_car']:+.1f} bps  median {R['eff_med']:+.1f}  "
      f"sd {R['eff_sd']:.1f}  hit {R['eff_hit']}%")
print(f"            cross-event t {R['eff_t']:+.2f}  HAC t (daily AR) {R['eff_hac']:+.2f}  "
      f"placebo p {R['eff_p']:.3f}")
print(f"\nevent bootstrap CI  announce [{R['ann_boot_lo']:+d}, {R['ann_boot_hi']:+d}]  "
      f"effective [{R['eff_boot_lo']:+d}, {R['eff_boot_hi']:+d}]  (bps)")
print(f"block bootstrap CI  announce [{R['ann_blk_lo']:+d}, {R['ann_blk_hi']:+d}]  "
      f"effective [{R['eff_blk_lo']:+d}, {R['eff_blk_hi']:+d}]  (bps)")

announce  : n=7  CAR -14.1 bps  median -22.8  sd 122.9  hit 43%
            cross-event t -0.30  HAC t (daily AR) -0.30  placebo p 0.802
            placebo null -3.4 +/- 64.2 bps, 95% band [-129, +124]
effective : n=7  CAR +35.7 bps  median +36.9  sd 81.6  hit 57%
            cross-event t +1.16  HAC t (daily AR) +0.86  placebo p 0.541

event bootstrap CI  announce [-93, +73]  effective [-15, +96]  (bps)
block bootstrap CI  announce [-108, +79]  effective [-42, +114]  (bps)


## 2. Power, stated before the conclusion

With n = 7 and a cross-event CAR dispersion of 123 bps, the smallest pooled CAR callable at 5% is ~114 bps; the placebo band agrees at ±~127 bps. This bounds what the null result means — it rules out a large wrapper-level footprint, not a 20-40 bps one.

> 💡 *In plain words:* a small effect could have been here and we would never have seen it. We can only say a big one is absent.

In [3]:
print(f"minimum detectable CAR (5%, n={R['n_events']}): announce {R['ann_mde']} bps, "
      f"effective {R['eff_mde']} bps")
print(f"placebo 95% band (announce leg): [{R['ann_null_lo']:+d}, {R['ann_null_hi']:+d}] bps")

minimum detectable CAR (5%, n=7): announce 114 bps, effective 75 bps
placebo 95% band (announce leg): [-129, +124] bps


## 3. The window sweep — and the small-sample t-statistic trap

Nine windows; the four that span day 0 or earlier are descriptive (not actionable) and labelled so. The `[-5,-1]` row is the instructive failure: a naive cross-event *t* of −3.43, and a randomisation *p* of 0.45.

> 💡 *In plain words:* the *t*-test guesses how noisy the world is from seven numbers. The random-date test does not have to guess.

In [4]:
print(f"[-5,-1] descriptive: CAR {R['pre_car']:+.1f} bps  naive cross-event t {R['pre_t']:+.2f}  "
      f"-> placebo p {R['pre_p']:.3f}")
print(f"worst (smallest) placebo p across all {R['n_windows']} windows: {R['worst_p']:.3f}")
print(f"Bonferroni-adjusted over {R['n_windows']} windows: {R['bonf']:.3f} for every one of them")

[-5,-1] descriptive: CAR -33.2 bps  naive cross-event t -3.43  -> placebo p 0.451
worst (smallest) placebo p across all 9 windows: 0.423
Bonferroni-adjusted over 9 windows: 1.000 for every one of them


## 4. Jackknife and era cut — how fragile a seven-point mean is

Dropping one event at a time moves the pooled CAR anywhere between −48.5 and +9.7 bps — it changes **sign**. No sub-sample gets past |*t*| = 1.32. The era cut splits 3 against 4 observations and the halves disagree in sign; it is reported because the house style requires it, not because three points decide anything.

> 💡 *In plain words:* remove one event and the answer changes shape. That is the definition of a result you should not trade.

In [5]:
print(f"jackknife: pooled CAR ranges {R['jk_min']:+.1f} .. {R['jk_max']:+.1f} bps "
      f"(it changes sign); worst t {R['jk_min_t']:+.2f}")
print(f"era pre-2015 (n={R['era_e_n']}): CAR {R['era_e_car']:+.1f} bps, t {R['era_e_t']:+.2f} "
      f"<- n=3, decides nothing")
print(f"era 2015->   (n={R['era_l_n']}): CAR {R['era_l_car']:+.1f} bps, t {R['era_l_t']:+.2f}  "
      f"<- opposite sign")

jackknife: pooled CAR ranges -48.5 .. +9.7 bps (it changes sign); worst t -1.32
era pre-2015 (n=3): CAR -60.0 bps, t -1.23 <- n=3, decides nothing
era 2015->   (n=4): CAR +20.4 bps, t +0.28  <- opposite sign


## 5. The tradable arm — beta-hedged, costed, borrow- and financing-swept

Long 1 unit treated, short `beta` units of the sibling (beta from the same clean pre-event window), on at +1, off at +10. Cost `(1+beta) x 2 x cost_bps` per round trip, one-way x NAV; borrow on the short notional.

**The beta-hedged pair is not dollar-neutral, and this study does not pretend it is.** Long 1 against short `beta` leaves `1 - beta` units of NAV as a real net position — net long +0.52 on the share-class-ban event (beta 0.48), net short −0.23 on NDX 2023 (beta 1.23). Only the naive 1x/1x variant finances itself exactly. That residual is charged (or credited) at an assumed bill rate, which is what makes the reported number an excess-of-cash return instead of one merely called that. It is small — mean +0.52 bps at 200 bps/yr, swept 0 to 500 — and it is charged anyway.

The naive 1x/1x variant is shown because it is the trap: its positive number is unhedged market exposure (SPY's beta to IWM is 0.48-0.81), not a rule-change effect.

> 💡 *In plain words:* hedge properly and the trade loses before you pay anyone.

In [6]:
print(f"beta-hedged, announce leg : gross {R['trade_gross']:+.1f} bps  "
      f"net {R['trade_net']:+.1f} bps (t {R['trade_t']:+.2f})  win {R['trade_win']}%")
print(f"naive 1x/1x variant       : gross {R['naive_gross']:+.1f} bps  "
      f"net {R['naive_net']:+.1f} bps (t {R['naive_t']:+.2f})  <- unhedged beta")
print(f"beta-hedged, effective leg: gross {R['eff_trade_gross']:+.1f} bps  "
      f"net {R['eff_trade_net']:+.1f} bps (t {R['eff_trade_t']:+.2f})")
print(f"\nfinancing on the residual (1-beta) exposure at {R['fin_rate']} bps/yr: "
      f"mean {R['fin_charge']:+.2f} bps  (net {R['fin0_net']:+.1f} at 0 bps/yr, "
      f"{R['fin500_net']:+.1f} at 500)")
print(f"every cell of the cost x borrow sweep is negative: from {R['cost0_net']:+.1f} bps "
      f"at ZERO cost to {R['cost25_net']:+.1f} bps (t {R['cost25_t']:+.2f}) at the worst corner")
print('the only |t| >= 2 on the tradable arm says the trade reliably LOSES money')

beta-hedged, announce leg : gross -7.7 bps  net -29.3 bps (t -0.62)  win 43%
naive 1x/1x variant       : gross +46.2 bps  net +24.3 bps (t +0.51)  <- unhedged beta
beta-hedged, effective leg: gross +43.2 bps  net +21.8 bps (t +0.70)

financing on the residual (1-beta) exposure at 200 bps/yr: mean +0.52 bps  (net -28.8 at 0 bps/yr, -30.1 at 500)
every cell of the cost x borrow sweep is negative: from -8.2 bps at ZERO cost to -115.5 bps (t -2.30) at the worst corner
the only |t| >= 2 on the tradable arm says the trade reliably LOSES money


## 6. Deployed-capital race (excess-of-cash)

Park in BIL, overlay the pair only inside event windows. Sparse by construction — 50 live days out of 4,802 — so the Sharpe is read as a scale marker, not a portfolio statistic.

**BIL starts 2007-05-30, so 2 of the 7 announcement events fall outside the cash era and are DROPPED.** They are not mapped onto BIL's first session: an earlier build did exactly that, stacking two pre-2007 events' P&L into the first ten days of the cash series and inventing a −3.8% drawdown out of nothing. Both dropped events happen to be losers, which is why the overlay below reads positive while the full-sample per-event mean is −29.3 bps net. The sub-sample is the flattering one, and it still earns 3 bps a year.

> 💡 *In plain words:* nineteen years, five trades, a rounding error.

In [7]:
print(f"{R['dc_events']} events inside the BIL era ({R['dc_dropped']} dropped: pre-BIL), "
      f"{R['live_days']} live days of {R['total_days']:,}")
print(f"total excess {R['overlay_total']:+d} bps over nineteen years")
print(f"overlay excess-of-cash Sharpe {R['overlay_sharpe']:+.3f} (HAC t {R['overlay_t']:+.2f})")
print(f"reference: SPY excess-of-cash Sharpe {R['spy_sharpe']:+.3f} over the same BIL window")

5 events inside the BIL era (2 dropped: pre-BIL), 50 live days of 4,802
total excess +65 bps over nineteen years
overlay excess-of-cash Sharpe +0.187 (HAC t +0.37)
reference: SPY excess-of-cash Sharpe +0.542 over the same BIL window


## 7. Live synthetic control — the detector is powered and unbiased

**Synthetic, not the real tape.** Three pairs, twelve announcements, a planted +250 bps abnormal drift bled in over the ten sessions after each; then a matched null. The planted world must clear the placebo; the null must not.

In [8]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from methodology_shock import data, strategy as st
pl = st.synthetic_detect(*data.synthetic_panel(signal_strength=1.0, seed=919)[:2],
                         window=(1, 10), n_draws=400)
print('SYNTHETIC (never supports the real-tape stamp)')
print(f"  planted 250 bps: CAR {pl['mean_car_bps']:+.1f} bps, t {pl['t_cross_event']:+.2f}, "
      f"placebo p {pl['placebo_p']:.3f}, hit {pl['hit_rate']:.0%}")
nulls = np.array([
    st.run_event_study(*data.synthetic_panel(signal_strength=0.0, seed=919 + 7 * k)[:2],
                       window=(1, 10))['mean_car_bps']
    for k in range(6)
])
print(f"  null across 6 seeds: mean {nulls.mean():+.1f} bps, sd {nulls.std(ddof=1):.1f} bps "
      f"-> centred on zero")

SYNTHETIC (never supports the real-tape stamp)
  planted 250 bps: CAR +314.9 bps, t +3.21, placebo p 0.000, hit 92%


  null across 6 seeds: mean +17.4 bps, sd 53.5 bps -> centred on zero


## Verdict

- **Signal — None.** Pooled CAR -14.1 bps on the announcement leg (placebo *p* = 0.802) and +35.7 bps on the effective leg (*p* = 0.541) — **opposite signs**, both inside a null band of [-129, +124] bps. All 9 windows carry a Bonferroni-adjusted *p* of 1.000 (smallest raw *p* 0.423); the event bootstrap [-93, +73] and block bootstrap [-108, +79] both straddle zero; the eras disagree in sign; the drop-one jackknife swings the pooled CAR from -48.5 to +9.7 bps, through zero. Nothing clears |*t*| >= 2 on the real tape. The synthetic control recovers a planted 250 bps shock at +315 bps (*p* = 0.000) and stays centred on the null (+17.4 bps, sd 53.5, six seeds), so the flat result is the tape's, not the harness's. **Survivorship: none to name** — four continuously listed wrappers chosen ex ante by the index each tracks; the selection risk lives in the hand-assembled event list, which the jackknife, the window sweep and the placebo test all interrogate — and in which one date was found wrong and corrected in audit, moving the headline by 22 bps on its own.
- **Tradability — Mirage.** -7.7 bps gross, -29.3 bps net per event (*t* -0.62), 43% win rate; every cell of the cost x borrow sweep negative, including the free-trading corner (-8.2 bps). The deployed-capital overlay's +65 bps is 5 post-2007 trades of noise (Sharpe +0.19, HAC *t* +0.37, against SPY's +0.54) on a sub-sample that excludes the two biggest losers.
- **Power caveat, stated plainly.** Minimum detectable CAR ~114 bps. This design rules out a *large* wrapper-level methodology shock, not a small one. The tradable claim — nothing big enough to pay for the spread — is what the tape supports.

---

*Every real-tape number above is frozen from [`docs/results.md`](../docs/results.md) (Fingerprint `547814cd71c7`, as-of 2026-06-30), reproducible with `python examples/verify.py`. The only live cell is the synthetic control, and it is banner-labelled.*